# MD analysis of Abeta42-mutant pentamers

## 0. Files loading and preparation

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align, distances
from MDAnalysis.analysis.rms import RMSF
from MDAnalysis.analysis.dssp import DSSP
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
from MDAnalysis.analysis import pca as mda_pca
from MDAnalysis.analysis.dihedrals import Ramachandran

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Style set-up
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

# File paths
GRO_FILE = "./md.gro"  
XTC_FILE = "./md_clean.xtc"  

# Mutation posistion

MUT_POS = int(input())

# Trajectories loading
print("Loading GROMACS files...")
u = mda.Universe(GRO_FILE, XTC_FILE)
ref = mda.Universe(GRO_FILE, XTC_FILE)

print(f"System loaded:")
print(f"  - Atoms: {u.atoms.n_atoms}")
print(f"  - Residues: {u.atoms.n_residues}")
print(f"  - Frames: {u.trajectory.n_frames}")
print(f"  - Time step: {u.trajectory.dt} пс")
print(f"  - Total time: {u.trajectory.n_frames * u.trajectory.dt / 1000:.1f} нс")

## 1. RMSD

In [ ]:
print("Calculating RMSD...")

# Trajectory alingnment
aligner = align.AlignTraj(u, ref, select="protein and name CA", in_memory=False).run()

# RMSD calculation
R = rms.RMSD(u, ref,
             select="protein and name CA",
             groupselections=["protein and backbone",
                              "protein"]).run()

time_ns = R.results.rmsd[:, 1] / 1000  # ps -> ns

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(time_ns, R.results.rmsd[:, 2], label="Cα", linewidth=1, color='#2E86AB')
ax.plot(time_ns, R.results.rmsd[:, 3], label="Backbone", linewidth=0.8, alpha=0.7, color='#A23B72')
ax.plot(time_ns, R.results.rmsd[:, 4], label="All atoms", linewidth=0.7, alpha=0.5, color='#F18F01')

ax.set_xlabel("Time (ns)", fontsize=11)
ax.set_ylabel("RMSD (Å)", fontsize=11)
ax.set_title("RMSD of Aβ42-mutant pentamer relative to starting structure", fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("01_rmsd.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nRMSD statistics:")
print(f"  Cα (mean): {np.mean(R.results.rmsd[:, 2]):.2f} ± {np.std(R.results.rmsd[:, 2]):.2f} Å")
print(f"  Backbone (mean): {np.mean(R.results.rmsd[:, 3]):.2f} ± {np.std(R.results.rmsd[:, 3]):.2f} Å")

## 2. RMSF

In [ ]:
print("Calculating RMSF...")

# Trajectory alignment before RMSF
align.AlignTraj(u, ref, select="protein and name CA").run()

# RMSF calculation for Cα
calphas = u.select_atoms("protein and name CA")
rmsfer = RMSF(calphas).run()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(calphas.resids, rmsfer.results.rmsf, linewidth=1.5, color='#2E86AB')

ax.set_xlabel("Residue number", fontsize=11)
ax.set_ylabel("RMSF (Å)", fontsize=11)
ax.set_title("Per-residue RMSF of Aβ42-mutant pentamer", fontsize=12, fontweight='bold')

mean_rmsf = np.mean(rmsfer.results.rmsf)
ax.axhline(y=mean_rmsf, color='red', linestyle='--', linewidth=1.5,
           label=f"Mean = {mean_rmsf:.2f} Å")

# Mutation highlight
# Positions in pentamer
mutation_positions = [MUT_POS, MUT_POS + 42, MUT_POS + 42*2, MUT_POS + 42*3, MUT_POS + 42*4]
for mut_pos in mutation_positions:
    if mut_pos in calphas.resids:
        ax.axvline(x=mut_pos, color='green', linestyle=':', alpha=0.5, linewidth=1.5)

ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("02_rmsf.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nRMSF statistics:")
print(f"  Mean: {mean_rmsf:.2f} ± {np.std(rmsfer.results.rmsf):.2f} Å")
print(f"  Min: {np.min(rmsfer.results.rmsf):.2f} Å (residue {calphas.resids[np.argmin(rmsfer.results.rmsf)]})")
print(f"  Max: {np.max(rmsfer.results.rmsf):.2f} Å (residue {calphas.resids[np.argmax(rmsfer.results.rmsf)]})")

## 3. Gyration radius

In [ ]:
print("Calculating radius of gyration...")

protein = u.select_atoms("protein")
rg_values = []
times = []

for ts in u.trajectory:
    rg_values.append(protein.radius_of_gyration())
    times.append(ts.time / 1000)  # ps -> ns

rg_values = np.array(rg_values) / 10  # Å -> nm
times = np.array(times)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Time series
axes[0].plot(times, rg_values, linewidth=0.7, alpha=0.8, color='#2E86AB')
axes[0].set_xlabel("Time (ns)", fontsize=11)
axes[0].set_ylabel("Rg (nm)", fontsize=11)
axes[0].set_title("Radius of gyration over time", fontsize=12, fontweight='bold')
axes[0].axhline(np.mean(rg_values), color='red', linestyle='--', linewidth=1.5, label=f'Mean = {np.mean(rg_values):.3f} nm')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Distribution
axes[1].hist(rg_values, bins=50, density=True, color='#2E86AB', edgecolor='white', alpha=0.7)
axes[1].set_xlabel("Rg (nm)", fontsize=11)
axes[1].set_ylabel("Probability density", fontsize=11)
axes[1].set_title(f"Rg distribution", fontsize=12, fontweight='bold')
axes[1].axvline(np.mean(rg_values), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(rg_values):.3f} nm')
axes[1].axvline(np.median(rg_values), color='green', linestyle=':', linewidth=2, label=f'Median = {np.median(rg_values):.3f} nm')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("03_radius_of_gyration.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nRadius of gyration statistics:")
print(f"  Mean: {np.mean(rg_values):.3f} ± {np.std(rg_values):.3f} nm")
print(f"  Median: {np.median(rg_values):.3f} nm")
print(f"  Range: [{np.min(rg_values):.3f}, {np.max(rg_values):.3f}] nm")
print(f"  StDev: {np.std(rg_values):.3f} nm")

## 4. DSSP analysis (Secondary structure)

In [ ]:
print("Calculating secondary structure with DSSP...")

dssp_analysis = DSSP(u).run()

# dssp_ndarray shape: (n_frames, n_residues, 3)
# Index 0 = loop, 1 = helix, 2 = sheet
dssp_data = dssp_analysis.results.dssp_ndarray

# Propensity calculus for each residue
helix_propensity = dssp_data[:, :, 1].mean(axis=0)
sheet_propensity = dssp_data[:, :, 2].mean(axis=0)
coil_propensity = dssp_data[:, :, 0].mean(axis=0)

protein_res = u.select_atoms("protein").residues
resids = protein_res.resids[:len(helix_propensity)]

fig, ax = plt.subplots(figsize=(13, 6))
width = 1
ax.bar(resids, helix_propensity, width=width, label="Helix", color="#E63946", alpha=0.8)
ax.bar(resids, sheet_propensity, width=width, bottom=helix_propensity,
       label="Sheet", color="#457B9D", alpha=0.8)
ax.bar(resids, coil_propensity, width=width,
       bottom=helix_propensity + sheet_propensity,
       label="Coil", color="#CCCCCC", alpha=0.6)

# Mutation highlight
mutation_positions = [MUT_POS, MUT_POS + 42, MUT_POS + 42*2, MUT_POS + 42*3, MUT_POS + 42*4]
for mut_pos in mutation_positions:
    if mut_pos in resids:
        ax.axvline(x=mut_pos, color='green', linestyle=':', linewidth=1.5, alpha=0.7)

ax.set_xlabel("Residue number", fontsize=11)
ax.set_ylabel("Fraction", fontsize=11)
ax.set_title("Secondary structure propensity of Aβ42-mutant pentamer", fontsize=12, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.set_ylim([0, 1])
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("04_secondary_structure.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nSecondary structure (mean within frames):")
print(f"  Helix: {np.mean(helix_propensity)*100:.1f}%")
print(f"  Sheet: {np.mean(sheet_propensity)*100:.1f}%")
print(f"  Coil: {np.mean(coil_propensity)*100:.1f}%")

## 5. Cα-Cα distance map

In [ ]:
print("Calculating average Cα–Cα distance map...")

ca_atoms = u.select_atoms("protein and name CA")
n_res = len(ca_atoms)

print(f"Total residues: {n_res}")

# Calculating average distance map (every 10th frame)
avg_dist = np.zeros((n_res, n_res))
n_frames_sampled = 0

for ts in u.trajectory[::10]:
    dist_array = distances.self_distance_array(ca_atoms.positions)
    dist_matrix = np.zeros((n_res, n_res))
    triu = np.triu_indices(n_res, k=1)
    dist_matrix[triu] = dist_array
    dist_matrix += dist_matrix.T
    np.fill_diagonal(dist_matrix, 0)
    avg_dist += dist_matrix
    n_frames_sampled += 1

avg_dist /= n_frames_sampled

print(f"Frames sampled: {n_frames_sampled}")

fig, ax = plt.subplots(figsize=(10, 9))

im = ax.imshow(
    avg_dist,
    cmap="YlOrRd_r",
    origin="lower",
    aspect="equal"
)

ax.set_xlabel("Residue index", fontsize=12)
ax.set_ylabel("Residue index", fontsize=12)
ax.set_title(
    "Average Cα–Cα distance map of Aβ42 F20L pentamer",
    fontsize=13,
    fontweight="bold"
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Distance (Å)", fontsize=11)

monomer_size = 42
n_monomers = n_res // monomer_size

for i in range(1, n_monomers):
    boundary = i * monomer_size - 0.5
    ax.axvline(
        x=boundary,
        color="white",
        linestyle="--",
        linewidth=0.7,
        alpha=0.8
    )
    ax.axhline(
        y=boundary,
        color="white",
        linestyle="--",
        linewidth=0.7,
        alpha=0.8
    )

tick_positions = np.arange(
    monomer_size / 2,
    n_res,
    monomer_size
)
tick_labels = [f"M{i+1}" for i in range(n_monomers)]
ax.set_xticks(tick_positions)
ax.set_yticks(tick_positions)
ax.set_xticklabels(tick_labels)
ax.set_yticklabels(tick_labels)

plt.tight_layout()
plt.savefig(
    "05_average_distance_map.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 6. Hydrogen bonds analysis

In [ ]:
print("Calculating hydrogen bonds...")

protein = u.select_atoms("protein")

hbonds = HydrogenBondAnalysis(
    universe=u,
    donors_sel="protein and name N",
    hydrogens_sel="protein and name H HN", 
    acceptors_sel="protein and name O",   
    d_a_cutoff=3.5,                      
    d_h_a_angle_cutoff=120           
)

hbonds.run()

# H-bonds number in every frame
frames = np.unique(hbonds.results.hbonds[:, 0])
hb_count = np.array([np.sum(hbonds.results.hbonds[:, 0] == f) for f in frames])
time_hb = frames * u.trajectory.dt / 1000  # ps -> ns

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(time_hb, hb_count, linewidth=0.8, alpha=0.8, color='#2E86AB')
ax.fill_between(time_hb, hb_count, alpha=0.3, color='#2E86AB')

ax.set_xlabel("Time (ns)", fontsize=11)
ax.set_ylabel("Number of intrapeptide H-bonds", fontsize=11)
ax.set_title(f"Backbone H-bonds in Aβ42-mutant pentamer (mean = {np.mean(hb_count):.1f})",
                 fontsize=12, fontweight='bold')
ax.axhline(np.mean(hb_count), color='red', linestyle='--', linewidth=1.5, label='Mean')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("06_hbonds.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nH-bonds statistics (backbone only):")
print(f"  Mean: {np.mean(hb_count):.1f} ± {np.std(hb_count):.1f}")
print(f"  Range: [{np.min(hb_count)}, {np.max(hb_count)}]")

## 7. PCA analysis

In [ ]:
print("Running PCA (Principal Component Analysis)...")

pc = mda_pca.PCA(u, select="protein and name CA", align=True).run()

# Trajectories' projections on first 3 PCs
backbone = u.select_atoms("protein and name CA")
transformed = pc.transform(backbone, n_components=3)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Cumulated variance
axes[0].plot(np.arange(1, 21), pc.results.cumulated_variance[:20] * 100, 'o-',
            color='#2E86AB', linewidth=2, markersize=6)
axes[0].set_xlabel("Principal component", fontsize=11)
axes[0].set_ylabel("Cumulated variance (%)", fontsize=11)
axes[0].set_title("Variance explained by top 20 PCs", fontsize=12, fontweight='bold')
axes[0].axhline(y=80, color='red', linestyle='--', linewidth=1.5, label='80% threshold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Projection
scatter = axes[1].scatter(transformed[:, 0], transformed[:, 1],
                          c=np.arange(len(transformed)), cmap="viridis",
                          s=5, alpha=0.6)
axes[1].set_xlabel("PC1 (Å)", fontsize=11)
axes[1].set_ylabel("PC2 (Å)", fontsize=11)
axes[1].set_title("Projection onto PC1 vs PC2", fontsize=12, fontweight='bold')
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label("Frame", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("07_pca_analysis.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nPCA results:")
print(f"  First 3 PCs explain {pc.results.cumulated_variance[2]*100:.1f}% variance")
print(f"  First 10 Pcs explain {pc.results.cumulated_variance[9]*100:.1f}% variance")

## 8. Ramachandran plot

In [ ]:
print("Calculating Ramachandran plot...")

protein_sel = u.select_atoms("protein")
rama = Ramachandran(protein_sel).run()
    
fig, ax = plt.subplots(figsize=(8, 8))
rama.plot(ax=ax, color='#2E86AB', marker='.', markersize=1.5, ref=True)
ax.set_title("Ramachandran plot - Aβ42-mutant pentamer ensemble", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("08_ramachandran.png", dpi=300, bbox_inches='tight')
plt.show()

## 9. Convergence assessment

In [ ]:
print("Convergence assessment...")

# Method: Rg
def block_average(data, n_blocks):
    block_size = len(data) // n_blocks
    block_means = [np.mean(data[i*block_size:(i+1)*block_size])
                   for i in range(n_blocks)]
    return np.mean(block_means), np.std(block_means) / np.sqrt(n_blocks)

# Compare first half and second half
half = len(rg_values) // 2
rg_first = rg_values[:half]
rg_second = rg_values[half:]

print(f"\nStep 1: Compare first and second half of the trajectory")
print(f"  Rg (first half):  {np.mean(rg_first):.3f} ± {np.std(rg_first):.3f} nm")
print(f"  Rg (second half):  {np.mean(rg_second):.3f} ± {np.std(rg_second):.3f} nm")
print(f"  Difference: {abs(np.mean(rg_first) - np.mean(rg_second)):.4f} nm")

# Method 2: Running average
window = min(500, len(rg_values) // 10)  # Adaptive window size
running_avg = np.convolve(rg_values, np.ones(window)/window, mode='valid')
times_running = times[:len(running_avg)]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Running average
axes[0, 0].plot(times_running, running_avg, linewidth=1.5, color='#2E86AB')
axes[0, 0].set_xlabel("Time (ns)", fontsize=10)
axes[0, 0].set_ylabel("Running average Rg (nm)", fontsize=10)
axes[0, 0].set_title(f"Convergence: Rg running average (window = {window} frames)", fontsize=11, fontweight='bold')
axes[0, 0].axhline(np.mean(rg_values), color='red', linestyle='--', linewidth=1.5, label="Overall mean")
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Block averaging
block_sizes = [10, 20, 50, 100, 200, 500, 1000]
errors = []
for bs in block_sizes:
    if bs <= len(rg_values):
        _, se = block_average(rg_values, len(rg_values)//bs if bs < len(rg_values) else 1)
        errors.append(se)
    else:
        break

axes[0, 1].plot(block_sizes[:len(errors)], errors, 'o-', linewidth=2, markersize=8, color='#2E86AB')
axes[0, 1].set_xlabel("Block size (frames)", fontsize=10)
axes[0, 1].set_ylabel("Standard error of Rg (nm)", fontsize=10)
axes[0, 1].set_title("Block averaging convergence", fontsize=11, fontweight='bold')
axes[0, 1].set_xscale('log')
axes[0, 1].grid(True, alpha=0.3, which='both')

# Compare first and second half
labels = ['First half', 'Second half']
means = [np.mean(rg_first), np.mean(rg_second)]
stds = [np.std(rg_first), np.std(rg_second)]
axes[1, 0].bar(labels, means, yerr=stds, capsize=10, color=['#A23B72', '#F18F01'], alpha=0.7, edgecolor='black')
axes[1, 0].set_ylabel("Rg (nm)", fontsize=10)
axes[1, 0].set_title("Rg comparison: first vs second half", fontsize=11, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Rg distribution for both halves
axes[1, 1].hist(rg_first, bins=30, alpha=0.6, label='First half', color='#A23B72', edgecolor='white')
axes[1, 1].hist(rg_second, bins=30, alpha=0.6, label='Second half', color='#F18F01', edgecolor='white')
axes[1, 1].set_xlabel("Rg (nm)", fontsize=10)
axes[1, 1].set_ylabel("Frequency", fontsize=10)
axes[1, 1].set_title("Rg distribution comparison", fontsize=11, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("09_convergence_assessment.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nStep 2: Block averaging")
print(f"  The standard error should stabilize as block size increases")
print(f"  (a plateau indicates good convergence)")

## 10. Summary

In [ ]:
print("\n" + "="*70)
print("MOLECULAR DYNAMICS ANALYSIS SUMMARY")
print("="*70)

print(f"\n1. STRUCTURAL STABILITY:")
print(f"   - Mean RMSD (Cα): {np.mean(R.results.rmsd[:, 2]):.2f} ± {np.std(R.results.rmsd[:, 2]):.2f} Å")
print(f"   - RMSD stabilized: {'YES' if np.mean(R.results.rmsd[-100:, 2]) < 1.3*np.mean(R.results.rmsd[:100, 2]) else 'NO'}")

print(f"\n2. COMPACTNESS:")
print(f"   - Mean Rg: {np.mean(rg_values):.3f} ± {np.std(rg_values):.3f} nm")
print(f"   - Rg range: [{np.min(rg_values):.3f}, {np.max(rg_values):.3f}] nm")

print(f"\n3. FLEXIBILITY:")
print(f"   - Mean RMSF: {np.mean(rmsfer.results.rmsf):.2f} ± {np.std(rmsfer.results.rmsf):.2f} Å")
print(f"   - Most flexible residues: N‑terminus and C‑terminus (as expected for an IDP)")

print(f"\n4. SECONDARY STRUCTURE:")
print(f"   - α‑helix fraction: {np.mean(helix_propensity)*100:.1f}%")
print(f"   - β‑sheet fraction: {np.mean(sheet_propensity)*100:.1f}%")
print(f"   - Loop/coil fraction: {np.mean(coil_propensity)*100:.1f}%")

print(f"\n5. CONVERGENCE:")
diff_rg = abs(np.mean(rg_first) - np.mean(rg_second))
combined_se = np.sqrt(np.std(rg_first)**2 + np.std(rg_second)**2) / np.sqrt(min(len(rg_first), len(rg_second)))
converged = diff_rg < combined_se
print(f"   - Rg first half: {np.mean(rg_first):.3f} ± {np.std(rg_first):.3f} nm")
print(f"   - Rg second half: {np.mean(rg_second):.3f} ± {np.std(rg_second):.3f} nm")
print(f"   - Convergence status: {'GOOD' if converged else 'NEEDS CHECKING'}")

print(f"\n" + "="*70)
print("MUTATION INFORMATION:")
print("="*70)
print(f"Mutation: Phe20 → Leu20 (F20L)")
print(f"Positions in the pentamer: 20, 62, 104, 146, 188")
print(f"Characteristics:")
print(f"  - Phenylalanine (original): aromatic, hydrophobic, benzene ring")
print(f"  - Leucine (new): aliphatic, hydrophobic, branched side chain")
print(f"\nExpected effects of the F20L mutation:")
print(f"  - Possible alteration of hydrophobic contacts (less aromatic interaction)")
print(f"  - Possible change in compactness due to different side‑chain volume")
print(f"  - Impact on secondary structure in the region of the mutation site")
print(f"  - Changes in dynamics around the mutation sites")

print(f"\n" + "="*70)
print("All plots saved in the current directory with prefix '0X_'")
print("="*70)